<a href="https://colab.research.google.com/github/tsarangler/ECON3916-Statistical-Machine-Learning/blob/main/Project%201/%5BProject_1%5D_Phase_3_The_Identification_Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3.1: The Baseline Model (OLS Implementation)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

url = 'https://raw.githubusercontent.com/tsarangler/ECON3916-Statistical-Machine-Learning/refs/heads/main/data/njmin3.csv'
df = pd.read_csv(url)
df.head()

,nj,d,d_nj,fte,bk,kfc,roys,wendys,co_owned,centralj,southj,pa1,pa2,demp
0,1,0,0,15.00,1,0,0,0,0,1,0,0,0,12.00
1,1,0,0,15.00,1,0,0,0,0,1,0,0,0,6.50
2,1,0,0,24.00,0,0,1,0,0,1,0,0,0,-1.00
3,1,0,0,19.25,0,0,1,0,1,0,0,0,0,2.25
4,1,0,0,21.50,1,0,0,0,0,0,0,0,0,13.00


In [4]:
# Define the formula string: Y ~ D + X1 + X2
formula_1 = 'fte ~ d + nj + bk + kfc + roys + co_owned'

# Fit the model using HC1 robust standard errors
model_1 = smf.ols(formula=formula_1, data=df).fit(cov_type='HC1')

# Print the "Regression Anatomy"
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:                    fte   R-squared:                       0.193
Model:                            OLS   Adj. R-squared:                  0.187
Method:                 Least Squares   F-statistic:                     56.72
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           2.76e-58
Time:                        16:21:06   Log-Likelihood:                -2822.1
No. Observations:                 794   AIC:                             5658.
Df Residuals:                     787   BIC:                             5691.
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.7575      1.080     21.992      0.0

# Step 3.2: Testing Heterogeneity (Interaction Terms)

In [5]:
# Interaction Term Syntax (:)
# This model includes: d (time), nj (state), AND (d * nj) = d_nj
# Testing if the minimum wage law (d) affects NJ restaurants differently than PA

formula_interact = 'fte ~ d * nj + bk + kfc + roys + co_owned'

# Fit the model with HC1 robust standard errors
model_interact = smf.ols(formula=formula_interact, data=df).fit(cov_type='HC1')

print(model_interact.summary())

                            OLS Regression Results                            
Dep. Variable:                    fte   R-squared:                       0.196
Model:                            OLS   Adj. R-squared:                  0.189
Method:                 Least Squares   F-statistic:                     48.14
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           5.96e-57
Time:                        16:22:40   Log-Likelihood:                -2820.4
No. Observations:                 794   AIC:                             5657.
Df Residuals:                     786   BIC:                             5694.
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     24.8875      1.352     18.402      0.0

# Step 3.3: The "Identification Police" Meeting

## The Story
The Card & Krueger (1994) study examines whether the 1992 New Jersey
minimum wage increase ($4.25 → $5.05) reduced fast food employment.
Standard labor economics predicts that raising the price of labor
(minimum wage) should reduce the quantity demanded (employment).
However, in a monopsonistic labor market, where employers have wage-setting
power, a minimum wage increase can actually increase employment by
correcting the market distortion. We use Pennsylvania as our control group
since it did not raise its minimum wage, allowing us to isolate the
causal effect of the NJ policy.

## The Bias Check (Omitted Variable Bias)
I wish I had local unemployment rate at the county level. This variable
likely correlates with both the treatment (NJ's decision to raise the
minimum wage was partly driven by labor market conditions) and our outcome
(FTE employment). Its absence likely biases our d_nj coefficient, though
the direction is ambiguous.

## The Interpretation (Ceteris Paribus)
The d:nj (DiD) coefficient of 2.8451 means that the minimum
wage increase was associated with a 2.8451  unit change in full-time
equivalent employment in New Jersey restaurants relative to Pennsylvania,
holding chain type and ownership constant.